# Selección y partición de series BCS-DBT

**Ejemplo reproducible del procedimiento de la metodología.** Se parte de los metadatos originales de entrenamiento de BCS-DBT para seleccionar series, dividir pacientes y definir las posiciones relativas de los cortes. Las particiones generadas **no son necesariamente las mismas que se emplearon para obtener los resultados publicados en la tesis**.

Este notebook **no descarga ni procesa volúmenes DICOM**. Las cantidades finales representan **imágenes previstas**, sujetas a la disponibilidad y procesamiento correcto de cada volumen.

**Uso:** Ejecutar las celdas en orden. Colocar los archivos `BCS-DBT-file-paths-train-v2.xlsx` y `BCS-DBT-labels-train-v2.csv` (se acepta también un sufijo de copia como ` (1)`) junto al notebook. En Colab se solicitarán los archivos si no están presentes. Requiere `pandas`, `numpy`, `scikit-learn` y `openpyxl`.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Ubicación de los metadatos: carpeta actual o carga manual en Colab.
def localizar_metadatos():
    rutas = sorted(Path('.').glob('BCS-DBT-file-paths-train-v2*.xlsx'))
    etiquetas = sorted(Path('.').glob('BCS-DBT-labels-train-v2*.csv'))
    if not rutas or not etiquetas:
        try:
            from google.colab import files
        except ImportError as exc:
            raise FileNotFoundError(
                'Colocá ambos archivos originales de metadatos en esta carpeta.'
            ) from exc
        print('Seleccioná los dos archivos de metadatos de BCS-DBT.')
        files.upload()
        rutas = sorted(Path('.').glob('BCS-DBT-file-paths-train-v2*.xlsx'))
        etiquetas = sorted(Path('.').glob('BCS-DBT-labels-train-v2*.csv'))
    if not rutas or not etiquetas:
        raise FileNotFoundError('Falta uno o ambos archivos de metadatos.')
    return rutas[0], etiquetas[0]

archivo_rutas, archivo_etiquetas = localizar_metadatos()
rutas = pd.read_excel(archivo_rutas)
etiquetas = pd.read_csv(archivo_etiquetas)
print('Metadatos cargados:', archivo_rutas.name, 'y', archivo_etiquetas.name)

## 1. Selección inicial balanceada de series

In [ ]:
# Integrar rutas y etiquetas mediante identificadores comunes
datos = pd.merge(
    rutas,
    etiquetas,
    on=["PatientID", "StudyUID", "View"],
    how="inner",
    validate="one_to_one"
)

# Definir las clases binarias
datos["clase"] = (
    datos[["Actionable", "Benign", "Cancer"]]
    .max(axis=1)
    .astype(int)
)

# Seleccionar todas las series positivas
positivas = datos[datos["clase"] == 1].copy()

# Seleccionar igual cantidad de series negativas
negativas = datos[datos["clase"] == 0].sample(
    n=len(positivas),
    replace=False,
    random_state=42
)

# Construir la selección inicial balanceada
seleccion_inicial = pd.concat(
    [positivas, negativas],
    ignore_index=True
)

seleccion_inicial = seleccion_inicial.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

# Verificar cantidades
assert len(positivas) == 916 and len(negativas) == 916
assert not seleccion_inicial.duplicated(["PatientID", "StudyUID", "View"]).any()

print("Series positivas:", len(positivas))
print("Series negativas:", len(negativas))
print("Total seleccionado:", len(seleccion_inicial))
print("\nDistribución de clases:")
print(seleccion_inicial["clase"].value_counts().sort_index())

## 2. Partición por paciente (80/20 y luego 50/50 del 20 %)

La selección de series negativas utiliza la semilla 42. Para obtener las cantidades mínimas positivas de los escenarios, se buscan de manera determinista semillas para la **asignación de pacientes** y se conserva la primera válida. En los datos suministrados se encontró la semilla 2. Este procedimiento es una **reconstrucción ilustrativa**, no la asignación histórica.

In [ ]:
from sklearn.model_selection import train_test_split

# Cantidades mínimas de series positivas requeridas
objetivos = {
    "train": 726,
    "val": 84,
    "test": 83,
}

pacientes = seleccion_inicial["PatientID"].unique()

particion_encontrada = False

# Búsqueda reproducible de una partición válida
for semilla in range(10000):

    pacientes_train, pacientes_resto = train_test_split(
        pacientes,
        test_size=0.20,
        random_state=semilla,
        shuffle=True
    )

    pacientes_val, pacientes_test = train_test_split(
        pacientes_resto,
        test_size=0.50,
        random_state=semilla,
        shuffle=True
    )

    train = seleccion_inicial[
        seleccion_inicial["PatientID"].isin(pacientes_train)
    ].copy()

    val = seleccion_inicial[
        seleccion_inicial["PatientID"].isin(pacientes_val)
    ].copy()

    test = seleccion_inicial[
        seleccion_inicial["PatientID"].isin(pacientes_test)
    ].copy()

    n_train = (train["clase"] == 1).sum()
    n_val = (val["clase"] == 1).sum()
    n_test = (test["clase"] == 1).sum()

    if (
        n_train >= objetivos["train"]
        and n_val >= objetivos["val"]
        and n_test >= objetivos["test"]
    ):
        particion_encontrada = True

        split_train = train
        split_val = val
        split_test = test

        print("Partición válida encontrada.")
        print("Semilla utilizada:", semilla)
        break

if particion_encontrada:

    for nombre, conjunto in [
        ("Entrenamiento", split_train),
        ("Validación", split_val),
        ("Prueba", split_test)
    ]:
        print(f"\n{nombre}")
        print("Pacientes:", conjunto["PatientID"].nunique())
        print(conjunto["clase"].value_counts().sort_index())

    # Verificación de pacientes compartidos
    ids_train = set(split_train["PatientID"])
    ids_val = set(split_val["PatientID"])
    ids_test = set(split_test["PatientID"])

    assert ids_train.isdisjoint(ids_val)
    assert ids_train.isdisjoint(ids_test)
    assert ids_val.isdisjoint(ids_test)

    print("\nVerificación: sin pacientes compartidos.")

else:
    raise RuntimeError("No se encontró una partición válida en 10000 intentos.")

## 3. Selección de series de base

Se seleccionan 726/726 series en entrenamiento, 84/84 en validación y 83/83 en prueba. Estas series constituyen la base para planificar ambos escenarios.

In [ ]:
# =========================================================
# SELECCIÓN DE LA BASE COMÚN DE SERIES
# =========================================================

def seleccionar_series_exactas(df, n_pos, n_neg, semilla=42):
    positivas = df[df["clase"] == 1].sample(
        n=n_pos, random_state=semilla
    )

    negativas = df[df["clase"] == 0].sample(
        n=n_neg, random_state=semilla
    )

    return pd.concat(
        [positivas, negativas],
        ignore_index=True
    ).sample(frac=1, random_state=semilla).reset_index(drop=True)


base_train = seleccionar_series_exactas(split_train, 726, 726)
base_val = seleccionar_series_exactas(split_val, 84, 84)
base_test = seleccionar_series_exactas(split_test, 83, 83)

# Verificación
for nombre, conjunto in [
    ("Entrenamiento", base_train),
    ("Validación", base_val),
    ("Prueba", base_test)
]:
    print(f"\n{nombre}")
    print(conjunto["clase"].value_counts().sort_index())

## 4. Escenario desbalanceado: un corte central por serie

Se incorporan series negativas adicionales **de pacientes de la partición correspondiente**. Cada serie aporta como máximo una imagen central.

In [ ]:
# Construcción del escenario desbalanceado (3:1)

objetivos = {
    "Entrenamiento": (2178, 726),
    "Validación": (250, 84),
    "Prueba": (251, 83),
}

columnas_id = ["PatientID", "StudyUID", "View"]
ids_iniciales = pd.MultiIndex.from_frame(
    seleccion_inicial[columnas_id]
)

def construir_desbalanceado(base, n_neg, n_pos):
    positivas = base[base["clase"] == 1].copy()
    negativas_base = base[base["clase"] == 0].copy()

    assert len(positivas) == n_pos

    # Buscar series negativas adicionales de los mismos pacientes
    pacientes = set(base["PatientID"])

    candidatas = datos[
        datos["PatientID"].isin(pacientes) &
        (datos["clase"] == 0)
    ].copy()

    ids_candidatas = pd.MultiIndex.from_frame(
        candidatas[columnas_id]
    )

    candidatas = candidatas[
        ~ids_candidatas.isin(ids_iniciales)
    ]

    faltantes = n_neg - len(negativas_base)

    adicionales = candidatas.sample(
        n=faltantes,
        replace=False,
        random_state=42
    )

    resultado = pd.concat(
        [positivas, negativas_base, adicionales],
        ignore_index=True
    )

    return resultado.sample(
        frac=1,
        random_state=42
    ).reset_index(drop=True)


desb_train = construir_desbalanceado(base_train, 2178, 726)
desb_val = construir_desbalanceado(base_val, 250, 84)
desb_test = construir_desbalanceado(base_test, 251, 83)

for nombre, conjunto in [
    ("Entrenamiento", desb_train),
    ("Validación", desb_val),
    ("Prueba", desb_test)
]:
    print(f"\n{nombre}")
    print(conjunto["clase"].value_counts().sort_index())
    print("Total:", len(conjunto))

# Comprobar que no existan pacientes compartidos
assert set(desb_train["PatientID"]).isdisjoint(desb_val["PatientID"])
assert set(desb_train["PatientID"]).isdisjoint(desb_test["PatientID"])
assert set(desb_val["PatientID"]).isdisjoint(desb_test["PatientID"])

print("\nVerificación: sin pacientes compartidos.")

# Posición central y comprobaciones de unicidad de series.
for parte in (desb_train, desb_val, desb_test):
    parte["n_cortes"] = 1
    parte["posiciones"] = "[0.5]"
    assert not parte.duplicated(["PatientID", "StudyUID", "View"]).any()

## 5. Escenario balanceado: planificación de cortes

Las negativas aportan solo el corte central. Las positivas aportan dos cortes (25 % y 50 %) o tres (25 %, 50 % y 75 %), según la selección reproducible con semilla 42.

In [ ]:
# Asignación de cortes positivos para el escenario balanceado

import numpy as np
import pandas as pd

objetivos_positivos = {
    "train": 1597,
    "val": 200,
    "test": 200,
}

def asignar_cortes_positivos(base, objetivo, semilla=42):
    positivas = base[base["clase"] == 1].copy()

    n_series = len(positivas)
    n_tres_cortes = objetivo - 2 * n_series

    assert 0 <= n_tres_cortes <= n_series

    # Seleccionar aleatoriamente las series que aportan tres cortes
    rng = np.random.default_rng(semilla)

    indices_tres = rng.choice(
        positivas.index.to_numpy(),
        size=n_tres_cortes,
        replace=False
    )

    positivas["n_cortes"] = 2
    positivas.loc[indices_tres, "n_cortes"] = 3

    # Asignar las posiciones relativas de extracción
    positivas["posiciones"] = positivas["n_cortes"].apply(
        lambda n: [0.25, 0.50, 0.75]
        if n == 3 else [0.25, 0.50]
    )

    return positivas


positivas_bal_train = asignar_cortes_positivos(
    base_train, objetivos_positivos["train"]
)

positivas_bal_val = asignar_cortes_positivos(
    base_val, objetivos_positivos["val"]
)

positivas_bal_test = asignar_cortes_positivos(
    base_test, objetivos_positivos["test"]
)

for nombre, conjunto in [
    ("Entrenamiento", positivas_bal_train),
    ("Validación", positivas_bal_val),
    ("Prueba", positivas_bal_test)
]:
    print(f"\n{nombre}")
    print("Series con 2 cortes:", (conjunto["n_cortes"] == 2).sum())
    print("Series con 3 cortes:", (conjunto["n_cortes"] == 3).sum())
    print("Total de imágenes previstas:", conjunto["n_cortes"].sum())

In [ ]:
# Selección de series negativas para el escenario balanceado

def seleccionar_negativas_balanceado(conjunto, cantidad, semilla=42):
    negativas = conjunto[conjunto["clase"] == 0].sample(
        n=cantidad,
        replace=False,
        random_state=semilla
    ).copy()

    negativas["n_cortes"] = 1
    negativas["posiciones"] = [[0.50] for _ in range(len(negativas))]

    return negativas


negativas_bal_train = seleccionar_negativas_balanceado(
    desb_train, 1600
)

negativas_bal_val = seleccionar_negativas_balanceado(
    desb_val, 200
)

negativas_bal_test = seleccionar_negativas_balanceado(
    desb_test, 200
)

# Verificar las cantidades finales previstas
for nombre, positivas, negativas in [
    ("Entrenamiento", positivas_bal_train, negativas_bal_train),
    ("Validación", positivas_bal_val, negativas_bal_val),
    ("Prueba", positivas_bal_test, negativas_bal_test)
]:
    print(f"\n{nombre}")
    print("Imágenes negativas:", negativas["n_cortes"].sum())
    print("Imágenes positivas:", positivas["n_cortes"].sum())
    print("Total:", negativas["n_cortes"].sum() + positivas["n_cortes"].sum())

In [ ]:
# Construcción de las particiones del escenario balanceado

bal_train = pd.concat(
    [positivas_bal_train, negativas_bal_train],
    ignore_index=True
)

bal_val = pd.concat(
    [positivas_bal_val, negativas_bal_val],
    ignore_index=True
)

bal_test = pd.concat(
    [positivas_bal_test, negativas_bal_test],
    ignore_index=True
)

# Verificación de las cantidades de imágenes previstas
for nombre, conjunto in [
    ("Entrenamiento", bal_train),
    ("Validación", bal_val),
    ("Prueba", bal_test)
]:
    print(f"\n{nombre}")
    print(conjunto.groupby("clase")["n_cortes"].sum())
    print("Total:", conjunto["n_cortes"].sum())

# Verificación de la separación por pacientes
assert set(bal_train["PatientID"]).isdisjoint(bal_val["PatientID"])
assert set(bal_train["PatientID"]).isdisjoint(bal_test["PatientID"])
assert set(bal_val["PatientID"]).isdisjoint(bal_test["PatientID"])

print("\nVerificación correcta: sin pacientes compartidos.")

objetivos_balanceado = {
    "Entrenamiento": {0: 1600, 1: 1597},
    "Validación": {0: 200, 1: 200},
    "Prueba": {0: 200, 1: 200},
}
for nombre, conjunto in [("Entrenamiento", bal_train), ("Validación", bal_val), ("Prueba", bal_test)]:
    assert conjunto.groupby("clase")["n_cortes"].sum().to_dict() == objetivos_balanceado[nombre]
    assert not conjunto.duplicated(["PatientID", "StudyUID", "View"]).any()

## 6. Verificación conjunta y exportación de manifiestos

Se exportan **manifiestos de series y cortes previstos**. No se generan imágenes `.npy` ni se verifica la descarga y extracción DICOM. Los archivos se guardan en `selecciones_generadas/`, y **no es necesario publicarlos** junto con el notebook: se pueden regenerar al ejecutarlo.

In [ ]:
from pathlib import Path

escenarios = {
    "balanceado": (bal_train, bal_val, bal_test),
    "desbalanceado": (desb_train, desb_val, desb_test),
}
objetivos = {
    "balanceado": [(1600, 1597), (200, 200), (200, 200)],
    "desbalanceado": [(2178, 726), (250, 84), (251, 83)],
}
salida = Path("selecciones_generadas")
salida.mkdir(exist_ok=True)

for escenario, particiones in escenarios.items():
    ids_por_grupo = [set(df["PatientID"]) for df in particiones]
    assert ids_por_grupo[0].isdisjoint(ids_por_grupo[1])
    assert ids_por_grupo[0].isdisjoint(ids_por_grupo[2])
    assert ids_por_grupo[1].isdisjoint(ids_por_grupo[2])
    print(f"\nEscenario {escenario} — pacientes compartidos: 0")
    for (nombre, df, (neg_esperadas, pos_esperadas)) in zip(
        ("train", "val", "test"), particiones, objetivos[escenario]
    ):
        conteos = df.groupby("clase")["n_cortes"].sum().to_dict()
        assert conteos == {0: neg_esperadas, 1: pos_esperadas}, (escenario, nombre, conteos)
        assert not df.duplicated(["PatientID", "StudyUID", "View"]).any()
        df.to_csv(salida / f"{escenario}_{nombre}.csv", index=False)
        print(f"{nombre}: negativas={conteos[0]}, positivas={conteos[1]}, total={sum(conteos.values())}")

print("\nSe generaron seis manifiestos en:", salida.resolve())
print("Son cantidades previstas de imágenes, no DICOM procesados.")